# Seasonality and Dummy Variables

Seasonal regression adds indicator variables for repeated periods such as quarters or months. For a cycle with L seasons, use L - 1 dummy variables and leave one season as the baseline. Leaving out one season avoids perfect collinearity with the intercept. With quarterly data and Q1 as the baseline, define:

- $Q2_t = 1$ if period $t$ is quarter 2, and 0 otherwise.
- $Q3_t = 1$ if period $t$ is quarter 3, and 0 otherwise.
- $Q4_t = 1$ if period $t$ is quarter 4, and 0 otherwise.

Then a linear trend plus additive seasonal model is

$$y_t = \beta_0 + \beta_1 t + \beta_2 Q2_t + \beta_3 Q3_t + \beta_4 Q4_t + \epsilon_t.$$

Q1 is the baseline season because all three dummy variables are zero in Q1.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from checks import check_columns, check_no_missing

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True
bike = pd.read_csv(Path('data/trk50_bike_sales.csv'))
print(check_columns(bike, ['Year', 'Quarter', 'Time', 'Sales']))
bike.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(bike['Time'], bike['Sales'], marker='o')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Sales')
axes[0].set_title('TRK-50 quarterly sales')
for year, group in bike.groupby('Year'):
    axes[1].plot(group['Quarter'], group['Sales'], marker='o', label=f'Year {year}')
axes[1].set_xticks([1,2,3,4])
axes[1].set_xlabel('Quarter')
axes[1].set_ylabel('Sales')
axes[1].set_title('Seasonal display')
axes[1].legend()
plt.tight_layout()

The plot shows an increasing linear trend across years and a stable quarterly pattern: Q3 is consistently highest, Q1 is consistently lowest, and the seasonal swing is similar from year to year. That is the additive, or constant, seasonality case.

In [ ]:
bike = bike.assign(
    Q2=(bike['Quarter'] == 2).astype(int),
    Q3=(bike['Quarter'] == 3).astype(int),
    Q4=(bike['Quarter'] == 4).astype(int),
)
model = smf.ols('Sales ~ Time + Q2 + Q3 + Q4', data=bike).fit()
print(model.summary().tables[1])

Coefficient interpretation:

- Intercept: fitted Q1 value when time is 0. It anchors the equation but may not be directly meaningful if time 0 is outside the data.
- Time coefficient: average period-to-period trend after accounting for season.
- Q2, Q3, Q4 coefficients: expected difference from Q1 at the same time.

For this model, Q3 has the largest seasonal coefficient, so Q3 is the strongest season relative to Q1.

In [ ]:
future = pd.DataFrame({
    'Time': [17, 18, 19, 20],
    'Quarter': [1, 2, 3, 4],
})
future = future.assign(
    Q2=(future['Quarter'] == 2).astype(int),
    Q3=(future['Quarter'] == 3).astype(int),
    Q4=(future['Quarter'] == 4).astype(int),
)
pred = model.get_prediction(future).summary_frame(alpha=0.05)
future.join(pred[['mean', 'mean_ci_lower', 'mean_ci_upper', 'obs_ci_lower', 'obs_ci_upper']]).round(2)

The confidence interval estimates the mean response for that future season. The prediction interval estimates a single future observation and is wider because it includes future random error.

Transfer question: if the series had monthly data, how many dummy variables would you include when one month is the baseline?